# Uncovering Hidden Myths

This portfolio project demonstrates a complete pipeline for building and evaluating a Knowledge Graph (KG). It integrates semantic web logic, deterministic rules, and machine learning into a cohesive data science architecture.

## Project Architecture (LO5)
**LO5 (Design and implement architectures):** To integrate a number of technologies successfully, the system is built on a 4-layer architecture:
*   **Data Layer:** Extracting and cleaning heterogeneous tabular data using `pandas`.
*   **Semantic Layer:** Structuring the data formally using `rdflib` (RDF, RDFS, OWL).
*   **Reasoning Layer:** Automatically deducing new facts using the `owlrl` engine and custom Python rules.
*   **Machine Learning Layer:** Predicting missing links using Knowledge Graph Embeddings (TransE in `pykeen`).

In [2]:
import ast
import re
import urllib.parse
import networkx as nx
import pandas as pd
import owlrl
from pykeen.pipeline import pipeline
from pykeen.predict import predict_target
from pykeen.triples import TriplesFactory
from pyvis.network import Network
from rdflib import Graph, Literal, Namespace
from rdflib.namespace import OWL, RDF, RDFS

/home/julia-tomaszkiewicz/Dokumenty/Knowledge Graph/Mythology-Knowledge-Graph/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Data Loading and EDA
First, we load the raw datasets and replace all missing values with empty strings to simplify parsing. We perform a quick Exploratory Data Analysis (EDA) to check the distribution of character types and genders.

In [3]:
characters_df = pd.read_csv('characters.csv').fillna('')
edges_df = pd.read_csv('edges_full.csv').fillna('')

print("Unique character genders found in dataset:")
print(characters_df[characters_df['gender'] != '']['gender'].value_counts())

print("\nCharacter type distribution in dataset:")
print(characters_df[characters_df['type'] != '']['type'].value_counts())

Unique character genders found in dataset:
gender
male               684
female             370
male organism        6
female organism      6
trans man            2
hermaphroditism      1
intersex             1
Name: count, dtype: int64

Character type distribution in dataset:
type
Mythological Greek Character    748
Greek deity                      96
Greek nymph                      34
titan                            24
Okeanid                          22
Potamoi                          21
naiad                            18
Greek water deities              18
Greek primordial deity           13
Olympian god                     12
allegorical Greek deity          11
Giants                           10
daemon                            8
cyclops                           4
Anemoi                            4
centaur                           4
sibyl                             4
Pleiades                          4
satyr                             3
Hecatoncheires                   

/tmp/ipykernel_12660/555154125.py:2: DtypeWarning: Columns (0: section) have mixed types. Specify dtype option on import or set low_memory=False.
  edges_df = pd.read_csv('edges_full.csv').fillna('')


## Step 2: Defining the Ontology (T-Box)
**LO4 (Data Models) & LO2 (Logical Knowledge):** We construct the complete Semantic Web data model (T-Box) upfront. This serves as the schema for our entire project.
* We build a multi-level class hierarchy (Types -> Deity -> MythologicalEntity).
* To represent logical knowledge that uses **full recursion** (LO2), we explicitly define `descends_from` as a `TransitiveProperty`.
* We also define all symmetric interactions upfront, including basic interactions (`interacts_with`) and domain-specific rules we will extract later (`is_generational_enemy_of`, `shares_type_with`).

In [4]:
g = Graph()
MYTH = Namespace("http://example.org/mythology/")
g.bind("myth", MYTH)

def clean_uri(text):
    return urllib.parse.quote(str(text).strip().replace(" ", "_").replace('"', '').replace("'", ""))

# 1. Class Hierarchy
g.add((MYTH['Deity'], RDFS.subClassOf, MYTH['MythologicalEntity']))

deity_types = [
    'Greek deity', 'Greek primordial deity', 'allegorical Greek deity', 
    'Greek water deities', 'chthonic deities', 'Olympian god', 'titan'
]

for c_type in characters_df['type'].unique():
    if not c_type: continue
    type_uri = MYTH[clean_uri(c_type)]
    g.add((type_uri, RDFS.subClassOf, MYTH['Deity'] if c_type in deity_types else MYTH['MythologicalEntity']))

# 2. Base Properties
g.add((MYTH['resides_in'], RDF.type, OWL.ObjectProperty))
g.add((MYTH['has_gender'], RDF.type, OWL.DatatypeProperty))
g.add((MYTH['has_alias'], RDF.type, OWL.DatatypeProperty))

# 3. Recursive & Symmetric Properties (Reasoning Rules)
g.add((MYTH['descends_from'], RDF.type, OWL.ObjectProperty))
g.add((MYTH['descends_from'], RDF.type, OWL.TransitiveProperty))

g.add((MYTH['interacts_with'], RDF.type, OWL.ObjectProperty))
g.add((MYTH['interacts_with'], RDF.type, OWL.SymmetricProperty))

g.add((MYTH['is_generational_enemy_of'], RDF.type, OWL.ObjectProperty))
g.add((MYTH['is_generational_enemy_of'], RDF.type, OWL.SymmetricProperty))

g.add((MYTH['shares_type_with'], RDF.type, OWL.ObjectProperty))
g.add((MYTH['shares_type_with'], RDF.type, OWL.SymmetricProperty))

<Graph identifier=Nfa6d34e25e2b459180be498da7dbe416 (<class 'rdflib.graph.Graph'>)>

## Step 3: Populating the Graph (A-Box)
**LO7 (KG Creation):** We map basic tabular CSV datasets into the ontology. This represents the explicit, foundational knowledge before any advanced NLP extraction or reasoning is applied.

In [5]:
gender_mapping = {'male organism': 'male', 'female organism': 'female'}

for _, row in characters_df.iterrows():
    subj_uri = MYTH[clean_uri(row['name'])]

    if row['type']: g.add((subj_uri, RDF.type, MYTH[clean_uri(row['type'])]))
    if row['residence']: g.add((subj_uri, MYTH['resides_in'], MYTH[clean_uri(row['residence'])]))
    if row['gender']: 
        norm_gen = gender_mapping.get(row['gender'].strip(), row['gender'].strip())
        g.add((subj_uri, MYTH['has_gender'], Literal(norm_gen)))

for _, row in edges_df.iterrows():
    subj_uri = MYTH[clean_uri(row['source'])]
    g.add((subj_uri, MYTH['interacts_with'], MYTH[clean_uri(row['target'])]))

print(f"Basic Knowledge Graph successfully populated with {len(g)} triples.")

Basic Knowledge Graph successfully populated with 28912 triples.


## Step 4: Semantic Querying with SPARQL
**LO6 (Querying):** To derive insights for simpler questions and verify data consistency, we execute a structured SPARQL query to retrieve all Olympian gods and their residences.

In [6]:
SPARQL_QUERY = """
PREFIX myth: <http://example.org/mythology/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?character ?residence
WHERE {
    ?character rdf:type myth:Olympian_god .
    ?character myth:resides_in ?residence .
}
"""

print("Executing SPARQL Query:")
for row in g.query(SPARQL_QUERY):
    print(f"- {row.character.split('/')[-1]} resides in {row.residence.split('/')[-1]}")

Executing SPARQL Query:
- Aphrodite resides in Olympus
- Apollo resides in Olympus
- Athena resides in Olympus
- Demeter resides in Olympus
- Hera resides in Olympus
- Hestia resides in Olympus
- Zeus resides in Olympus


## Step 5: Advanced Data Cleaning (Aliases)
Before diving into domain logic, we perform advanced tabular data cleaning. We parse the `aliases` column (which contains stringified lists) into distinct relations. Because this does not require NLP or semantic reasoning, it will be included in our 'Clean ML' dataset.

In [7]:
clean_aliases = set()

for _, row in edges_df.iterrows():
    src = str(row['source']).strip()
    try:
        if '[' in row['source_aliases']:
            for alias in ast.literal_eval(row['source_aliases']):
                clean_aliases.add((src, 'has_alias', str(alias).strip().replace(" ", "_")))
    except:
        pass

df_aliases = pd.DataFrame(list(clean_aliases), columns=['Subject', 'Relation', 'Object'])
print(f"Successfully parsed {len(df_aliases)} alias connections.")

Successfully parsed 745 alias connections.


## Step 6: Domain-Specific Logic & NLP Extraction
**LO2 (Logical Knowledge) & LO7 (Creation):** Now we apply pure domain knowledge. First, we use Python rules to explicitly link generational enemies (Olympians vs. Titans) and group entities of the same specific type. Secondly, we use NLP (Regex) to scan unstructured text and extract `descends_from` lineages, adding them directly to the semantic graph.

In [8]:
new_facts = set()
generic_types = ['Mythological Greek Character', 'Greek deity']

for _, row in edges_df.iterrows():
    src, tgt = str(row['source']).strip(), str(row['target']).strip()
    s_type, t_type = str(row['source_type']).strip(), str(row['target_type']).strip()
    
    # Deterministic Python Rules
    if (s_type == 'Olympian god' and t_type == 'titan') or (s_type == 'titan' and t_type == 'Olympian god'):
        new_facts.add((src, 'is_generational_enemy_of', tgt))
        
    if s_type and s_type == t_type and s_type not in generic_types:
        new_facts.add((src, 'shares_type_with', tgt))

    # NLP Regex Extraction
    match = re.search(r'(?i)(?:son|daughter)\s+of\s+([A-Z][a-z]+)', str(row['source_description']))
    if match: 
        g.add((MYTH[clean_uri(src)], MYTH['descends_from'], MYTH[clean_uri(match.group(1))]))

df_advanced_logic = pd.DataFrame(list(new_facts), columns=['Subject', 'Relation', 'Object'])
descends_count = len(list(g.triples((None, MYTH['descends_from'], None))))

print(f"Deterministically extracted {len(df_advanced_logic)} python-based domain rules.")
print(f"Extracted {descends_count} explicit parent-child links via NLP.")

Deterministically extracted 670 python-based domain rules.
Extracted 369 explicit parent-child links via NLP.


## Step 7: Logic Engine (Reasoning & Recursion)
**LO6 (Scalable Reasoning) & LO8 (KG Evolution):** Now that the explicit NLP parent-child links are in the graph, we apply scalable reasoning using the OWL-RL engine. Because `descends_from` is a TransitiveProperty, the engine recursively infers all deep ancestral lineages.

In [9]:
pre_reasoned_triples = [
    [urllib.parse.unquote(s.split('/')[-1]), p.split('/')[-1], urllib.parse.unquote(o.split('/')[-1])]
    for s, p, o in g if p in (MYTH['descends_from'], MYTH['interacts_with'])
]
df_pre_reasoned = pd.DataFrame(pre_reasoned_triples, columns=['Subject', 'Relation', 'Object'])

size_before = len(g)
descends_before = len(list(g.triples((None, MYTH['descends_from'], None))))

owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g)

size_after = len(g)
descends_after = len(list(g.triples((None, MYTH['descends_from'], None))))

print(f"Semantic Engine inferred and autonomously added {size_after - size_before} new facts.")
print(f"Deep ancestral links recursively inferred by the engine: {descends_after - descends_before}")

Semantic Engine inferred and autonomously added 29455 new facts.
Deep ancestral links recursively inferred by the engine: 166


## Step 8: Architecture Integration (Ablation Study Setup)
**LO12 (Connections between KGs, ML and AI):** To quantitatively measure the exact impact of each logical component, we prepare FOUR distinct datasets for a rigorous Ablation Study:
1. **Baseline Graph:** Only raw, uncleaned edges (No contextual attributes).
2. **Clean ML Graph:** Basic attributes + cleaned aliases (No NLP/Reasoning).
3. **Domain & NLP Graph:** Clean ML + Python rules + explicit Regex extracted links (No OWL-RL recursive reasoning).
4. **Enriched Graph (Full Neuro-Symbolic):** The ultimate dataset incorporating everything above PLUS the OWL-RL inferences.

In [10]:
def format_ent(text):
    return str(text).strip().replace(" ", "_").replace('"', '').replace("'", "")

# 1. BASELINE GRAPH 
baseline_triples = []
for _, row in edges_df.iterrows():
    baseline_triples.append([format_ent(row['source']), 'interacts_with', format_ent(row['target'])])
df_baseline = pd.DataFrame(baseline_triples, columns=['Subject', 'Relation', 'Object']).drop_duplicates()

# 2. CLEAN ML GRAPH (Includes attributes and parsed aliases)
ml_triples = []
for _, row in edges_df.iterrows():
    ml_triples.append([format_ent(row['source']), 'interacts_with', format_ent(row['target'])])

for _, row in characters_df.iterrows():
    name = format_ent(row['name'])
    if row['type']: ml_triples.append([name, 'is_type', format_ent(row['type'])])
    if row['residence']: ml_triples.append([name, 'resides_in', format_ent(row['residence'])])
    if row['gender']: 
        ml_triples.append([name, 'has_gender', gender_mapping.get(row['gender'].strip(), row['gender'].strip())])
df_ml_base = pd.DataFrame(ml_triples, columns=['Subject', 'Relation', 'Object'])
df_ml = pd.concat([df_ml_base, df_aliases], ignore_index=True).drop_duplicates()

# 3. DOMAIN & NLP GRAPH (Before OWL-RL Reasoning)
df_domain_nlp = pd.concat([df_ml, df_advanced_logic, df_pre_reasoned], ignore_index=True).drop_duplicates()

# 4. ENRICHED GRAPH (Full Logic + NLP + OWL-RL Reasoning)
post_reasoned_triples = [
    [urllib.parse.unquote(s.split('/')[-1]), p.split('/')[-1], urllib.parse.unquote(o.split('/')[-1])]
    for s, p, o in g if p in (MYTH['descends_from'], MYTH['interacts_with'])
]
df_post_reasoned = pd.DataFrame(post_reasoned_triples, columns=['Subject', 'Relation', 'Object'])
df_enriched = pd.concat([df_ml, df_advanced_logic, df_post_reasoned], ignore_index=True).drop_duplicates()

tf_baseline = TriplesFactory.from_labeled_triples(df_baseline.values)
tf_ml = TriplesFactory.from_labeled_triples(df_ml.values)
tf_domain = TriplesFactory.from_labeled_triples(df_domain_nlp.values)
tf_enriched = TriplesFactory.from_labeled_triples(df_enriched.values)

print(f"1. Baseline Graph: {len(df_baseline)} edges.")
print(f"2. Clean ML Graph: {len(df_ml)} edges.")
print(f"3. Domain & NLP Graph (No Recursion): {len(df_domain_nlp)} edges.")
print(f"4. Enriched Graph (Full Neuro-Symbolic): {len(df_enriched)} edges.")

1. Baseline Graph: 26696 edges.
2. Clean ML Graph: 29618 edges.
3. Domain & NLP Graph (No Recursion): 30657 edges.
4. Enriched Graph (Full Neuro-Symbolic): 57519 edges.


## Step 9: Knowledge Graph Embeddings
**LO1 (KG Embeddings):** We train four separate translation-based TransE models to comprehensively trace how each layer of added intelligence (cleaning -> rules -> reasoning) impacts the sub-symbolic vector representations.

In [11]:
base_train, base_test = tf_baseline.split([0.8, 0.2], random_state=42)
ml_train, ml_test = tf_ml.split([0.8, 0.2], random_state=42)
domain_train, domain_test = tf_domain.split([0.8, 0.2], random_state=42)
enriched_train, enriched_test = tf_enriched.split([0.8, 0.2], random_state=42)

common_kwargs = dict(model='TransE', epochs=100, random_seed=42, training_kwargs=dict(batch_size=128))

print("Training 1/4: BASELINE model...")
base_result = pipeline(training=base_train, testing=base_test, **common_kwargs)

print("\nTraining 2/4: CLEAN ML model...")
ml_result = pipeline(training=ml_train, testing=ml_test, **common_kwargs)

print("\nTraining 3/4: DOMAIN & NLP model (Explicit Rules, No Reasoning)...")
domain_result = pipeline(training=domain_train, testing=domain_test, **common_kwargs)

print("\nTraining 4/4: ENRICHED model (Full OWL-RL Reasoning)...")
enriched_result = pipeline(training=enriched_train, testing=enriched_test, **common_kwargs)

No cuda devices were available. The model runs on CPU


Training 1/4: BASELINE model...


/home/julia-tomaszkiewicz/Dokumenty/Knowledge Graph/Mythology-Knowledge-Graph/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Training epochs on cpu: 100%|██████████| 100/100 [01:49<00:00,  1.09s/epoch, loss=0.175, prev_loss=0.176]
Evaluating on cpu:   0%|          | 0.00/5.34k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 5.34k/5.34k [00:02<00:00, 1.96ktriple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 2.84s seconds
INFO:pykeen.pipeline.api:Using device: None
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embe


Training 2/4: CLEAN ML model...


Training epochs on cpu: 100%|██████████| 100/100 [01:54<00:00,  1.15s/epoch, loss=0.105, prev_loss=0.106]
Evaluating on cpu:   0%|          | 0.00/5.92k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 5.92k/5.92k [00:05<00:00, 1.12ktriple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 5.40s seconds
INFO:pykeen.pipeline.api:Using device: None
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()



Training 3/4: DOMAIN & NLP model (Explicit Rules, No Reasoning)...


Training epochs on cpu: 100%|██████████| 100/100 [01:57<00:00,  1.18s/epoch, loss=0.103, prev_loss=0.101]
Evaluating on cpu:   0%|          | 0.00/6.13k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 6.13k/6.13k [00:05<00:00, 1.11ktriple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 5.66s seconds
INFO:pykeen.pipeline.api:Using device: None
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()



Training 4/4: ENRICHED model (Full OWL-RL Reasoning)...


Training epochs on cpu: 100%|██████████| 100/100 [03:17<00:00,  1.98s/epoch, loss=0.122, prev_loss=0.12]
Evaluating on cpu:   0%|          | 0.00/11.5k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 11.5k/11.5k [00:11<00:00, 982triple/s]  
INFO:pykeen.evaluation.evaluator:Evaluation took 11.96s seconds


## Step 10: Recommender System & Evaluation
**LO9 (Real-world Applications) & LO11 (Services):** We deploy the models as a real-world service—a KG-based recommender system utilizing both deep logic and KGE-based knowledge. We perform link prediction to suggest missing historical interactions and evaluate the results.

In [12]:
def get_metrics(result):
    m = result.metric_results
    return m.get_metric('mean_reciprocal_rank'), m.get_metric('mean_rank'), m.get_metric('hits@3'), m.get_metric('hits@10')

m_base = get_metrics(base_result)
m_ml = get_metrics(ml_result)
m_dom = get_metrics(domain_result)
m_enr = get_metrics(enriched_result)

print("\n========================= 4-WAY MODEL EVALUATION COMPARISON =========================")
print(f"{'Metric':<12} | {'1. Baseline':<12} | {'2. Clean ML':<12} | {'3. Domain+NLP':<15} | {'4. Enriched':<15}")
print("-" * 77)
print(f"{'MRR':<12} | {m_base[0]:<12.4f} | {m_ml[0]:<12.4f} | {m_dom[0]:<15.4f} | {m_enr[0]:<15.4f}")
print(f"{'Mean Rank':<12} | {m_base[1]:<12.1f} | {m_ml[1]:<12.1f} | {m_dom[1]:<15.1f} | {m_enr[1]:<15.1f}")
print(f"{'Hits@3':<12} | {m_base[2]:<12.2%} | {m_ml[2]:<12.2%} | {m_dom[2]:<15.2%} | {m_enr[2]:<15.2%}")
print(f"{'Hits@10':<12} | {m_base[3]:<12.2%} | {m_ml[3]:<12.2%} | {m_dom[3]:<15.2%} | {m_enr[3]:<15.2%}")
print("=====================================================================================")

QUERY_CHARACTER = "Zeus"
predictions = predict_target(model=enriched_result.model, head=QUERY_CHARACTER, relation="interacts_with", triples_factory=enriched_result.training)
df_results = predictions.df

known_interactions = set(edges_df[edges_df['source'] == QUERY_CHARACTER]['target'].tolist() + edges_df[edges_df['target'] == QUERY_CHARACTER]['source'].tolist())
df_results['is_known'] = df_results['tail_label'].apply(lambda x: x in known_interactions)

new_discoveries = df_results[(df_results['tail_label'] != QUERY_CHARACTER) & (~df_results['is_known'])]
print(f"\nTop 5 Novel Predicted Interactions for {QUERY_CHARACTER}:")
print(new_discoveries.head(5)[['tail_label', 'score']])


========================= 4-WAY MODEL EVALUATION COMPARISON =========================
Metric       | 1. Baseline  | 2. Clean ML  | 3. Domain+NLP   | 4. Enriched    
-----------------------------------------------------------------------------
MRR          | 0.1035       | 0.1099       | 0.1076          | 0.1556         
Mean Rank    | 88.7         | 101.6        | 101.8           | 56.7           
Hits@3       | 13.15%       | 13.83%       | 13.65%          | 22.77%         
Hits@10      | 29.78%       | 29.24%       | 29.41%          | 41.76%         

Top 5 Novel Predicted Interactions for Zeus:
            tail_label     score
855      Helen_of_Troy -9.188091
331    Augeias_of_Elis -9.322380
127     Ajax_the_Great -9.596858
274     Argus_Panoptes -9.811272
983  Ion_son_of_Xuthus -9.887211


## Step 11: Graph Visualization
Finally, we render an interactive network graph to visually inspect the clusters and connections in our enriched dataset.

In [ ]:
nx_graph = nx.Graph()
# number of edges limited to 2500, because my laptop is flying away loading this anyway
for i, row in df_enriched.head(2500).iterrows():
    nx_graph.add_edge(row['Subject'], row['Object'], title=row['Relation'])

net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", notebook=True, cdn_resources='remote')
net.from_nx(nx_graph)
net.repulsion(node_distance=150, spring_length=200)
net.show("mythology_graph.html")

mythology_graph.html
